In [1]:
from HMM import ChessHMM
from Utils import ChessUtils
import time

import torch
import numpy as np
import ChessLens

from matplotlib import pyplot as plt

/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


# Testing ChessHMM

In [2]:
fen = "rnbqkb1r/ppppp1pp/7n/4Pp2/8/8/PPPP1PPP/RNBQKBNR w KQkq f6 0 3"
pos = None

In [3]:
%%timeit
pos = ChessUtils.ChessTensorUtils.FENtoTensor(fen)

32.4 μs ± 4.53 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [4]:
%%timeit
pos = ChessHMM.ChessGameState(fen)

1.32 μs ± 210 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [6]:
fen = "rnbqkb1r/pppppBpp/7n/5p2/4P3/8/PPPP1PPP/RNBQK1NR b KQkq - 3 3"
pos = ChessHMM.ChessGameState(fen)
print(pos)
pos

rnbqkb-r
pppppBpp
-------n
-----p--
----P---
--------
PPPP-PPP
RNBQK-NR



FEN: rnbqkb1r/pppppBpp/7n/5p2/4P3/8/PPPP1PPP/RNBQK1NR b KQkq - 3 3

rnbqkb-r
pppppBpp
-------n
-----p--
----P---
--------
PPPP-PPP
RNBQK-NR

Turn: Black
Castling Rights: KQkq
En Passant: -
Check: True
Stalemate: False
Gameover: False

# Testing with ChessLens

In [2]:
img1 = ChessLens.ChessLensImage("/mnt/D/University/Thesis_Dataset/Temp/img_2.jpeg")
img1.detect_board()
img1.recognize_pieces()

img2 = ChessLens.ChessLensImage("/mnt/D/University/Thesis_Dataset/Temp/img_3.jpeg")
img2.detect_board()
img2.recognize_pieces()

In [12]:
def prep_piece_matrix(piece_matrix: torch.Tensor) -> np.ndarray:
    return -np.log(torch.rot90(piece_matrix, k=1, dims=(2,3)).squeeze().permute(1,2,0).numpy()[::-1]+(1e-7))

hmm = ChessHMM.ChessHMM(30)
start_time = time.perf_counter()
hmm.set_probs(1, prep_piece_matrix(img1.piece_matrix))
hmm.set_probs(2, prep_piece_matrix(img2.piece_matrix))
end_time = time.perf_counter()

hmm.bind(2)

f"{((end_time - start_time)/2) * 1e6} µs"

'1110.895000238088 µs'

In [18]:
hmm = ChessHMM.ChessHMM(20)

start_time = time.perf_counter()
for i in range(1, 121):
    hmm.set_probs(i, np.random.randn(8,8,13))
    if (i%30 == 0):
        hmm.bind(i-29)
end_time = time.perf_counter()

f"{((end_time-start_time)/120)*1e3} ms"

'1.1662437250076134 ms'

In [ ]:
%%timeit
hmm.get_history(True)

5.96 μs ± 51.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [22]:
print(hmm.print(1))

Prob: 65.859742

rnbqkbnr
pppppppp
--------
--------
--------
-----N--
PPPPPPPP
RNBQKB-R




Prob: 88.316187

rnbqkbnr
pppppppp
--------
--------
--------
-------N
PPPPPPPP
RNBQKB-R




Prob: 88.527027

rnbqkbnr
pppppppp
--------
--------
--------
--------
PPPPPPPP
RNBQKBNR




Prob: 97.942838

rnbqkbnr
pppppppp
--------
--------
--------
--P-----
PP-PPPPP
RNBQKBNR




Prob: 98.680323

rnbqkbnr
pppppppp
--------
--------
--------
--N-----
PPPPPPPP
R-BQKBNR







In [23]:
print(hmm.print(2))

Prob: 149.958365

rnbqkbnr
ppp-pppp
--------
---p----
--------
-----N--
PPPPPPPP
RNBQKB-R




Prob: 179.547454

rnbqkbnr
pppppppp
--------
--------
--------
-----N--
PPPPPPPP
RNBQKB-R




Prob: 179.547454

rnbqkbnr
ppp-pppp
---p----
--------
--------
-----N--
PPPPPPPP
RNBQKB-R




Prob: 189.491073

rnbqkbnr
pp-ppppp
--p-----
--------
--------
-----N--
PPPPPPPP
RNBQKB-R




Prob: 189.540033

rnbqkbnr
pp-ppppp
--------
--p-----
--------
-----N--
PPPPPPPP
RNBQKB-R





